# Inference Engineering — 2 Hour Task


## Task 1: Implement quantization from scratch


In [1]:
import numpy as np

def quantize_int8(values):
    """
    Quantize a float32 array to int8 using affine (zero-point) quantization.
    Returns (quantized_array, scale, zero_point).
    """
    arr = np.asarray(values, dtype=np.float32)

    qmin, qmax = -128, 127  # int8 range

    min_val = float(arr.min())
    max_val = float(arr.max())

    # Make sure the range includes 0 (needed so zero maps exactly to an int)
    min_val = min(min_val, 0.0)
    max_val = max(max_val, 0.0)

    # scale = how many float units per integer step
    scale = (max_val - min_val) / (qmax - qmin)
    if scale == 0:
        scale = 1e-8  # avoid divide-by-zero for constant arrays

    # zero_point = the integer that float 0.0 maps to
    zero_point = qmin - min_val / scale
    zero_point = int(round(zero_point))
    zero_point = max(qmin, min(qmax, zero_point))

    q = np.round(arr / scale + zero_point)
    q = np.clip(q, qmin, qmax).astype(np.int8)

    return q, scale, zero_point


def dequantize_int8(q, scale, zero_point):
    """Recover approximate float32 values from int8 + scale/zero_point."""
    return (q.astype(np.float32) - zero_point) * scale


In [3]:
# Test on a small array
test_values = np.array([-2.5, -1.0, 0.0, 1.5, 3.2], dtype=np.float32)

q, scale, zero_point = quantize_int8(test_values)
recovered = dequantize_int8(q, scale, zero_point)
error = np.abs(test_values - recovered)

print("Original :", test_values)
print("Quantized:", q)
print(f"Scale = {scale:.6f}, Zero-point = {zero_point}")
print("Recovered:", np.round(recovered, 4))
print("Abs error:", np.round(error, 4))
print(f"Max abs error: {error.max():.6f}, Mean abs error: {error.mean():.6f}")


Original : [-2.5 -1.   0.   1.5  3.2]
Quantized: [-128  -61  -16   51  127]
Scale = 0.022353, Zero-point = -16
Recovered: [-2.5035 -1.0059  0.      1.4976  3.1965]
Abs error: [0.0035 0.0059 0.     0.0024 0.0035]
Max abs error: 0.005882, Mean abs error: 0.003059


**Observations**: Int8 mein convert karke wapas original value mein convert karne par thoda sa quantization error aata hai. Ye error generally scale/2 ke around hota hai, jo upar diye gaye max absolute error mein bhi clearly dikh raha hai. Agar input values ka range (max−min) zyada hai, toh distortion bhi increase hota hai. Aisa isliye hota hai kyunki kuch outlier values ki wajah se int8 ki limited resolution properly use nahi ho pati. Isliye practical applications mein per-channel ya per-tensor scaling ka sahi choice karna important hota hai.

## Task 2: Estimate model size from parameter count

In [4]:
BYTES_PER_PARAM = {
    "fp32": 4,
    "fp16": 2,
    "int8": 1,
}

def model_size(num_params, dtype="fp32"):
    """
    Return (size_in_MB, size_in_GB) for a model with `num_params` parameters
    stored in the given dtype.
    """
    if dtype not in BYTES_PER_PARAM:
        raise ValueError(f"Unsupported dtype: {dtype}. Choose from {list(BYTES_PER_PARAM)}")

    total_bytes = num_params * BYTES_PER_PARAM[dtype]
    size_mb = total_bytes / (1024 ** 2)
    size_gb = total_bytes / (1024 ** 3)
    return size_mb, size_gb


In [5]:
example_models = {
    "125M": 125_000_000,
    "1B":   1_000_000_000,
    "7B":   7_000_000_000,
}

print(f"{'Model':<8}{'dtype':<8}{'Size (MB)':>14}{'Size (GB)':>14}")
print("-" * 44)
for name, params in example_models.items():
    for dtype in ["fp32", "fp16", "int8"]:
        mb, gb = model_size(params, dtype)
        print(f"{name:<8}{dtype:<8}{mb:>14,.1f}{gb:>14,.2f}")
    print()


Model   dtype        Size (MB)     Size (GB)
--------------------------------------------
125M    fp32             476.8          0.47
125M    fp16             238.4          0.23
125M    int8             119.2          0.12

1B      fp32           3,814.7          3.73
1B      fp16           1,907.3          1.86
1B      int8             953.7          0.93

7B      fp32          26,702.9         26.08
7B      fp16          13,351.4         13.04
7B      int8           6,675.7          6.52



**Observations:** FP32 se INT8 mein jaane par memory requirement 4x reduce ho jaati hai. Iska main reason ye hai ki FP32 mein ek parameter ke liye 4 bytes use hote hain, jabki INT8 mein sirf 1 byte lagta hai. 7B model ke liye memory around 26 GB (FP32) se reduce hokar 6.5 GB (INT8) ho jaati hai. Isliye quantization ek bahut useful technique hai, especially jab humein large models ko smaller hardware par deploy karna ho.

## Task 3: System design scenario — multi-GPU inference



**Scenario:** Serving a large language model that doesn't fit on a single GPU, on a node with 8 GPUs connected
via NVLink, and multiple such nodes connected via InfiniBand.

### Design note

**Where to split the model**

- **Within a node (NVLink domain): tensor parallelism.** Split individual layers (attention heads, MLP matrices)
  across the 8 GPUs in a node. Tensor parallelism requires an all-reduce (or all-gather/reduce-scatter) after
  *every* layer, which means very frequent, latency-sensitive, high-bandwidth communication. NVLink's high
  bandwidth (hundreds of GB/s per link) and low latency make this affordable; doing it over a slower link would
  stall every layer.

- **Across nodes (InfiniBand domain): pipeline parallelism (and/or data parallelism).** Split the model
  depth-wise into stages, with each node holding a contiguous chunk of layers. Communication between stages is
  just the activation tensors passed at stage boundaries — infrequent relative to tensor-parallel communication,
  and it can be overlapped with compute using micro-batching (pipelining). InfiniBand's lower bandwidth compared
  to NVLink is tolerable here because there's far less data to move and more opportunity to hide the latency.

- If the workload also needs more throughput (not just to fit the model), add **data parallelism** across
  replicas of the whole pipeline, again communicating over InfiniBand (e.g., for gradient/optimizer-state
  sync in training, or independent request batches in inference).

**Role of NVLink vs InfiniBand**

- **NVLink** = the *intra-node* fabric. High bandwidth, low latency, direct GPU-to-GPU. It's meant for
  fine-grained, frequent synchronization — exactly what tensor parallelism needs (an all-reduce per layer,
  sometimes many times per forward pass).
- **InfiniBand** = the *inter-node* fabric. Lower bandwidth than NVLink and higher latency (extra network hops,
  NICs, switches), but scales to many nodes. It's meant for coarse-grained communication — pipeline stage
  hand-offs, or periodic parameter synchronization — where communication volume and frequency are much lower.

**What would go wrong if the roles were swapped**

- **Tensor parallelism over InfiniBand:** every layer would now wait on a slow, higher-latency all-reduce across
  nodes. Since tensor-parallel communication is on the critical path many times per forward pass, GPUs would sit
  idle waiting for the network constantly — the job becomes network-bound and throughput collapses, even though
  the GPUs themselves are fast.
- **Coarse pipeline hand-offs confined to NVLink only (never crossing to InfiniBand):** this under-uses NVLink's
  strength (it's overkill for infrequent activation hand-offs) and doesn't solve the original problem — once the
  model exceeds one node's GPUs you must cross node boundaries somewhere, so InfiniBand is needed regardless.
  The real failure mode is specifically putting the *frequent, bandwidth-hungry* pattern (tensor parallelism) on
  the *slower* link — that mismatch is what tanks performance.

**Summary:** match communication pattern to link characteristics — frequent/high-bandwidth (tensor parallel)
goes on NVLink within a node; infrequent/coarser (pipeline/data parallel) goes on InfiniBand across nodes.


## Task 4: Compute-bound vs memory-bound estimation
Uses a roofline-style comparison: time to do the compute vs. time to move the required bytes. Whichever is larger determines the bottleneck.

In [6]:
def bound_type(flops_per_token, bytes_per_token, compute_flops_per_sec, mem_bandwidth_bytes_per_sec):
    """
    Determine whether inference for one token is compute-bound or memory-bound.

    flops_per_token             : FLOPs required to produce one output token
    bytes_per_token              : bytes that must be read from memory per token
                                    (e.g. model weights read during autoregressive decode)
    compute_flops_per_sec        : GPU peak compute throughput (FLOPs/sec)
    mem_bandwidth_bytes_per_sec  : GPU memory bandwidth (bytes/sec)

    Returns a dict with the verdict and the two time estimates.
    """
    time_compute = flops_per_token / compute_flops_per_sec
    time_memory = bytes_per_token / mem_bandwidth_bytes_per_sec

    if time_compute > time_memory:
        verdict = "compute-bound"
    elif time_memory > time_compute:
        verdict = "memory-bound"
    else:
        verdict = "balanced (roofline ridge point)"

    # Arithmetic intensity = FLOPs per byte moved; useful diagnostic
    arithmetic_intensity = flops_per_token / bytes_per_token
    ridge_point = compute_flops_per_sec / mem_bandwidth_bytes_per_sec

    return {
        "verdict": verdict,
        "time_compute_s": time_compute,
        "time_memory_s": time_memory,
        "arithmetic_intensity_flops_per_byte": arithmetic_intensity,
        "ridge_point_flops_per_byte": ridge_point,
    }


In [7]:
# Example configs
# GPU spec roughly modeled on an A100-class card: ~312 TFLOPs (fp16 tensor), ~2 TB/s HBM bandwidth
A100_COMPUTE = 312e12       # FLOPs/sec
A100_MEM_BW  = 2.0e12       # bytes/sec

configs = {
    # Small model, single-token autoregressive decode (batch size 1):
    # ~2 FLOPs per parameter per token (standard transformer estimate), bytes read ~= model size in fp16
    "1B model, batch=1 decode": dict(
        flops_per_token=2 * 1e9,
        bytes_per_token=1e9 * 2,   # fp16 -> 2 bytes/param, whole model read per token
        compute_flops_per_sec=A100_COMPUTE,
        mem_bandwidth_bytes_per_sec=A100_MEM_BW,
    ),
    # Large model, single-token decode
    "70B model, batch=1 decode": dict(
        flops_per_token=2 * 70e9,
        bytes_per_token=70e9 * 2,
        compute_flops_per_sec=A100_COMPUTE,
        mem_bandwidth_bytes_per_sec=A100_MEM_BW,
    ),
    # Same 70B model but large batch (batched decode): weight read amortized across many tokens in the batch
    "70B model, batch=256 decode": dict(
        flops_per_token=2 * 70e9,
        bytes_per_token=(70e9 * 2) / 256,
        compute_flops_per_sec=A100_COMPUTE,
        mem_bandwidth_bytes_per_sec=A100_MEM_BW,
    ),
}

for name, cfg in configs.items():
    result = bound_type(**cfg)
    print(f"--- {name} ---")
    for k, v in result.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.6g}")
        else:
            print(f"  {k}: {v}")
    print()


--- 1B model, batch=1 decode ---
  verdict: memory-bound
  time_compute_s: 6.41026e-06
  time_memory_s: 0.001
  arithmetic_intensity_flops_per_byte: 1
  ridge_point_flops_per_byte: 156

--- 70B model, batch=1 decode ---
  verdict: memory-bound
  time_compute_s: 0.000448718
  time_memory_s: 0.07
  arithmetic_intensity_flops_per_byte: 1
  ridge_point_flops_per_byte: 156

--- 70B model, batch=256 decode ---
  verdict: compute-bound
  time_compute_s: 0.000448718
  time_memory_s: 0.000273438
  arithmetic_intensity_flops_per_byte: 256
  ridge_point_flops_per_byte: 156



**Observations:** Batch size 1 par decoding mainly **memory-bound** hoti hai. GPU ko har token ke liye complete model weights HBM se read karne padte hain, isliye **quantization ya weight compression** zyada helpful hoti hai, kyunki isse move hone wala data kam ho jata hai.

Jab batch size bada ho jata hai, jaise **256**, toh same weights multiple tokens ke liye use hote hain. Is wajah se memory read ka cost divide ho jata hai aur workload **compute-bound** ho jata hai. Is case mein **faster matmuls, better kernels aur more FLOPs** zyada useful hote hain.

Basically, **batching aur speculative decoding** jaise techniques inference ko memory-bound se compute-bound side shift karne mein help karti hain, jisse GPU ki computing power ka better use hota hai.
